# PoseLab: reproduce the research comparison

This notebook runs the unchanged pretrained **CoST-GCN\*** baseline and the project's pose-recovery extension on the same real Kinetics-skeleton clips. It does not contain example or invented results: output cells start empty.

Base paper: [Continual Spatio-Temporal Graph Convolutional Networks](https://doi.org/10.1016/j.patcog.2023.109528), *Pattern Recognition* 140 (2023), 109528. [Authors' repository](https://github.com/LukasHedegaard/continual-skeletons).

In Google Colab, choose a CPU or available GPU runtime. Free runtime availability and duration vary. A GPU is useful if the benchmark supports it, but the checkpoint can also run on CPU. Keep this notebook running until evaluation finishes.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repository = Path('/content/miniproject-2') if Path('/content').exists() else Path.cwd() / 'miniproject-2'
if not repository.exists():
    subprocess.run(['git', 'clone', 'https://github.com/4uashish2628/miniproject-2.git', str(repository)], check=True)
os.chdir(repository)
print('Project:', repository)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


## Install the recorded dependencies

Run this once per fresh runtime. The authors' old training framework is not installed; the project adapter uses their pinned neural-network definitions and verified pretrained checkpoint.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-research.txt'], check=True)


## Verify the source and checkpoint

Setup pins the authors' source to commit `a8fe2937a33f24cce65c1f8c2fc41081bceda721` and verifies the 25.9 MB checkpoint by SHA256. An existing changed checkout or mismatched checkpoint stops setup instead of being overwritten.


In [ ]:
subprocess.run([sys.executable, 'scripts/setup_research.py'], check=True)


## Download a fixed real-data pilot

The dataset archive is about 7.47 GB, but this downloader fetches only its index and 600 selected JSON pose sequences using bounded HTTP byte ranges: approximately **52 MB transferred** and **98 MB of selected raw JSON** for the fixed seed below. It does not download the full archive or RGB videos.

Sampling uses the SHA256 ordering of `2026:<clip_id>` over all 19,906 official validation clips. The first 200 form a calibration set; the next 400 are a disjoint test set. No clips are selected because the model recognizes them correctly. The original classifier retains all 400 outputs. These sampled sets contain 168 and 251 activity classes respectively, so this is a pilot, not a reproduction of the full published benchmark. Two empty test clips are retained and recorded in the manifest.


In [ ]:
subprocess.run([
    sys.executable, 'scripts/research_data.py',
    '--seed', '2026', '--calibration', '200', '--test', '400',
], check=True)

import json
manifest = json.loads(Path('.build/research-data/manifest.json').read_text())
for key in ['candidate_count', 'calibration_count', 'test_count', 'calibration_classes', 'test_classes', 'empty_skeleton_count']:
    print(f'{key}: {manifest[key]}')


## Calibrate and evaluate

`--phase all` runs calibration before evaluation. Thresholds and repair settings must be chosen using calibration clips only; the test set remains reserved for the final comparison. Both methods receive identical corrupted copies of each clip. Results should be interpreted as measured differences under this protocol, not as a guaranteed accuracy gain or a claim of worldwide novelty.

For an interrupted run, inspect `python -m research.benchmark --help`. The separate `calibrate` and `evaluate` phases allow evaluation after a completed calibration without selecting settings on test results.


In [ ]:
subprocess.run([sys.executable, '-m', 'research.benchmark', '--phase', 'all'], check=True)

subprocess.run([sys.executable, 'scripts/render_research_report.py'], check=True)


## Inspect and download the generated results

Only files actually produced by the benchmark are displayed and packaged. Keep the results together with the data manifest and project revision when reporting the experiment. Accuracy after rejecting uncertain predictions must always be read alongside coverage; rejecting more clips alone does not demonstrate better recognition.


In [ ]:
from IPython.display import HTML, Image, display

results = Path('results/research')
if not results.exists():
    raise FileNotFoundError('No results/research directory was produced; inspect the benchmark output above.')
for path in sorted(results.rglob('*')):
    if path.is_file():
        print(path, path.stat().st_size, 'bytes')
html_reports = sorted(results.rglob('*.html'))
if html_reports:
    display(HTML(filename=str(html_reports[0])))
else:
    for plot in sorted(results.rglob('*.png'))[:3]:
        display(Image(filename=str(plot)))


In [ ]:
import zipfile

archive_path = Path('/content/PoseLab_Research_Results.zip') if Path('/content').exists() else repository / 'PoseLab_Research_Results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(results.rglob('*')):
        if path.is_file():
            bundle.write(path, arcname=path.as_posix())
    bundle.write('.build/research-data/manifest.json', arcname='data_manifest.json')
    revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    bundle.writestr('project_revision.txt', revision + '\n')
print('Created:', archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Download the ZIP from this notebook environment file browser.')
